#### Etape 1.3 : Agregations avancees Spark
- Joindre consommations avec referentiel batiments
- Calculer l'intensite energetique (kWh/m2)
- Identifier les batiments hors norme (>3x la mediane de leur categorie)
- Calculer les totaux par commune et par type de batiment
- Creer une vue SQL exploitable

In [5]:
import os

DATA_DIR = "../data" #WHAT ?
DATA_DIR = os.path.join(DATA_DIR, "..", "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "..", "output", "consommation_clean")

CONSOMMATION_PATH = os.path.join(DATA_DIR, "consommations_raw.csv")
BATIMENTS_PATH = os.path.join(DATA_DIR, "batiments.csv")

In [6]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ECF2 - AGGREG") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

#moins de logs
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.7
Spark UI: http://host.docker.internal:4041


Récupération des données (data des bâtiments déjà incluses dans le fichier parquet dans l'étape précédente)

In [8]:
import pandas as pd


df_pandas = pd.read_parquet(OUTPUT_DIR)

df = spark.createDataFrame(df_pandas)

print(df.printSchema())

root
 |-- batiment_id: string (nullable = true)
 |-- unite: string (nullable = true)
 |-- conso_clean: double (nullable = true)
 |-- hour: long (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- type: string (nullable = true)
 |-- commune: string (nullable = true)
 |-- surface_m2: long (nullable = true)
 |-- annee_construction: long (nullable = true)
 |-- classe_energetique: string (nullable = true)
 |-- nb_occupants_moyen: long (nullable = true)
 |-- date: string (nullable = true)
 |-- type_energie: string (nullable = true)

None


#### Calcul de l'intensité énergétique (Consommation énergétique (kWh)​ / Surface (m²))

In [14]:
print("Calcul de la conso énergétique (conso/surface) ")

# root
#  |-- batiment_id: string (nullable = true)
#  |-- unite: string (nullable = true)
#  |-- conso_clean: double (nullable = true)
#  |-- hour: long (nullable = true)
#  |-- year: long (nullable = true)
#  |-- month: long (nullable = true)
#  |-- nom: string (nullable = true)
#  |-- type: string (nullable = true)
#  |-- commune: string (nullable = true)
#  |-- surface_m2: long (nullable = true)
#  |-- annee_construction: long (nullable = true)
#  |-- classe_energetique: string (nullable = true)
#  |-- nb_occupants_moyen: long (nullable = true)
#  |-- date: string (nullable = true)
#  |-- type_energie: string (nullable = true)


df_elec = (
    df
    .filter(F.col("type_energie") == "electricite")
    .filter(F.col("unite") == "kWh")
)

df_elec = df_elec.groupBy(
    "month", "batiment_id", "surface_m2", "commune", "type"
    ).agg(
        (F.sum("conso_clean") / F.col("surface_m2")).alias("conso_energy")
    )


df_elec.show()

windows = Window.partitionBy()

df_abberants = (
    df_elec
    .withColumn(
        "median",
        F.percentile_approx("conso_energy", 0.5).over(windows)
    )
    .filter(F.col("conso_energy") > 3 * F.col("median"))
    .drop("median")
)

df_abberants.show()


Calcul de la conso énergétique (conso/surface) 
+-----+-----------+----------+-------------+-----------+------------------+
|month|batiment_id|surface_m2|      commune|       type|      conso_energy|
+-----+-----------+----------+-------------+-----------+------------------+
|    1|    BAT0003|      1695|        Paris|      ecole|114.62279056047201|
|    1|    BAT0010|      1240|         Lyon|mediatheque|253.95233064516088|
|    1|    BAT0084|      1365|  Montpellier|    gymnase|238.06513553113538|
|    1|    BAT0015|      1393|         Lyon|     mairie| 139.6980114860015|
|    1|    BAT0089|      2276|         Nice|    gymnase|365.34176625659046|
|    1|    BAT0090|      2038|         Nice|    gymnase| 186.3216830225712|
|    1|    BAT0022|      1343|    Marseille|     mairie| 166.8293298585257|
|    1|    BAT0023|      1426|    Marseille|      ecole|229.58260869565206|
|    1|    BAT0028|      1878|    Marseille|      ecole|193.71528221512253|
|    1|    BAT0029|      2022|    Marsei

Calculer les totaux par commune et par type de batiment

In [16]:
df_abberants.groupBy("commune", "type").agg(
    F.count("*").alias("nb_aberrants")
)

df_abberants.show()

+-----+-----------+----------+-------------+-------+------------------+
|month|batiment_id|surface_m2|      commune|   type|      conso_energy|
+-----+-----------+----------+-------------+-------+------------------+
|    1|    BAT0099|      1798|       Rennes|piscine| 632.2962680756398|
|    1|    BAT0072|      3797|   Strasbourg|piscine| 476.2428469844615|
|    1|    BAT0019|      3120|         Lyon|piscine| 627.4495608974357|
|    1|    BAT0048|      3754|        Lille|piscine| 732.7303436334579|
|    1|    BAT0134|      2126|Saint-Etienne|piscine|  699.279887111947|
|    2|    BAT0048|      3754|        Lille|piscine| 677.5448454981351|
|    2|    BAT0072|      3797|   Strasbourg|piscine|  441.407613905715|
|    2|    BAT0121|      2639|     Le Havre|piscine| 668.9993747631678|
|    1|    BAT0005|      3913|        Paris|piscine| 729.0305724508047|
|    1|    BAT0146|      2609|       Toulon|piscine| 620.3306975852814|
|    1|    BAT0118|      3924|     Le Havre|piscine|484.49315494

#### Créer une vue exploitable

In [18]:
df_abberants.createOrReplaceTempView("aberrants")

spark.sql("""
                        SELECT
                            batiment_id,
                            commune,
                            conso_energy
                        FROM aberrants
                        ORDER BY conso_energy DESC
                        LIMIT 10
""").show()

+-----------+-------------+-----------------+
|batiment_id|      commune|     conso_energy|
+-----------+-------------+-----------------+
|    BAT0121|     Le Havre|741.5154604016675|
|    BAT0043|     Bordeaux|739.9206099166299|
|    BAT0043|     Bordeaux|738.7014699429573|
|    BAT0112|        Reims|736.2293990306945|
|    BAT0112|        Reims|734.7115347334407|
|    BAT0048|        Lille|732.7303436334579|
|    BAT0048|        Lille| 732.243100692594|
|    BAT0122|     Le Havre|731.0032362254591|
|    BAT0005|        Paris|729.0305724508047|
|    BAT0134|Saint-Etienne|726.7926669802442|
+-----------+-------------+-----------------+

